<a href="https://colab.research.google.com/github/Kylekk29/ImageAIUpscaler/blob/main/ImageAIUpscaler.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
#@title 1. ⚙️ Install Real-ESRGAN Engine (Updated with Bugfix)
#@markdown Click the "Play" button on the left to set up the environment.

import os
import site
from IPython.display import clear_output

print("⏳ Installing Real-ESRGAN and downloading weights... Please wait.")

# Clone repo
!git clone https://github.com/xinntao/Real-ESRGAN.git > /dev/null 2>&1
%cd Real-ESRGAN

# Install dependencies
!pip install -q basicsr facexlib gfpgan
!pip install -q -r requirements.txt
!python setup.py develop > /dev/null 2>&1

# Download the main AI weights
!wget -q https://github.com/xinntao/Real-ESRGAN/releases/download/v0.1.0/RealESRGAN_x4plus.pth -P weights/

# --- AUTOMATED BUGFIX START ---
for path in site.getsitepackages():
    target_file = os.path.join(path, 'basicsr/data/degradations.py')
    if os.path.exists(target_file):
        with open(target_file, 'r') as f:
            code = f.read()
        code = code.replace('functional_tensor', 'functional')
        with open(target_file, 'w') as f:
            f.write(code)
# --- AUTOMATED BUGFIX END ---

clear_output()
print("✅ Setup complete and bug patched! Proceed to Step 2.")

📦 Installing opencv-python...
/content/Real-ESRGAN
🔄 Clearing PyTorch CUDA cache...

📁 Select your image(s):


KeyboardInterrupt: 

In [5]:
#@title 2. 🚀 Quick-Enhance Panel (Gigapixel, Metadata Sync, Sharpness & Fast Mode)
upscale_factor = 6 #@param {type:"slider", min:2, max:32, step:2}
tile_size = 512 #@param {type:"slider", min:0, max:2048, step:128}
enhance_faces = False #@param {type:"boolean"}
opencv_method = "Lanczos4" #@param ["Lanczos4", "Bicubic", "Linear"]
apply_sharpness = True #@param {type:"boolean"}
sharpness_intensity = 0.6 #@param {type:"slider", min:0.1, max:2.0, step:0.1}
max_edge_resolution = "8K" #@param ["4K", "8K", "12K", "16K", "No Limit"]
output_folder = "/content/output" #@param {type:"string"}

import sys
import subprocess
import warnings
warnings.filterwarnings('ignore')

# Install required packages with proper error handling
required_packages = ['pillow-heif', 'piexif', 'opencv-python']
for pkg in required_packages:
    pkg_name = pkg.replace('-', '_')
    if pkg_name not in sys.modules:
        print(f"📦 Installing {pkg}...")
        try:
            subprocess.run([sys.executable, "-m", "pip", "install", pkg, "-q"],
                          check=True, capture_output=True)
        except subprocess.CalledProcessError as e:
            print(f"⚠️ Failed to install {pkg}, trying alternative method...")
            subprocess.run([sys.executable, "-m", "pip", "install", pkg, "--no-deps", "-q"])

import os
import glob
import shutil
import torch
import gc
import cv2
import numpy as np
from google.colab import files
from PIL import Image
import pillow_heif
import piexif
from pathlib import Path

# Optimize PyTorch settings
torch.backends.cudnn.benchmark = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

# Register HEIF support
pillow_heif.register_heif_opener()
Image.MAX_IMAGE_PIXELS = None

# Change to Real-ESRGAN directory
try:
    %cd /content/Real-ESRGAN
except Exception as e:
    print(f"⚠️ Error changing directory: {e}")
    print("Cloning Real-ESRGAN repository...")
    subprocess.run(["git", "clone", "https://github.com/xinntao/Real-ESRGAN.git", "/content/Real-ESRGAN"])
    %cd /content/Real-ESRGAN
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt", "-q"])

# Clear CUDA cache
print(f"🔄 Clearing PyTorch CUDA cache...")
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

# Clean and create directories
for f in ['inputs', 'results', output_folder]:
    if os.path.exists(f):
        try:
            shutil.rmtree(f)
        except Exception as e:
            print(f"⚠️ Could not remove {f}: {e}")
    os.makedirs(f, exist_ok=True)

print("\n📁 Select your image(s):")
uploaded = files.upload()

if not uploaded:
    print("⚠️ Upload cancelled. Exiting...")
else:
    print("\n📊 --- STAGE 1: DISK-STREAM INGESTION & METADATA CACHE ---")

    metadata_cache = {}
    processed_files = []
    original_filenames = {}  # Map output names to original names

    for idx, (name, file_bytes) in enumerate(uploaded.items(), 1):
        try:
            clean_name = "".join(c for c in name if c.isalnum() or c in '._- ')
            clean_name = clean_name.replace(" ", "_")
            target_path = os.path.join('inputs', clean_name)

            # Save uploaded file
            with open(target_path, 'wb') as f_out:
                f_out.write(file_bytes)

            # Handle HEIC/HEIF files
            ext = os.path.splitext(clean_name)[1].lower()
            if ext in ['.heic', '.heif']:
                jpg_path = os.path.splitext(target_path)[0] + '.jpg'
                with Image.open(target_path) as img_obj:
                    if "exif" in img_obj.info:
                        metadata_cache[os.path.basename(jpg_path)] = img_obj.info["exif"]
                    img_obj.convert('RGB').save(jpg_path, 'JPEG', quality=95, optimize=True)
                os.remove(target_path)
                target_path = jpg_path
                processed_files.append(target_path)
                original_filenames[os.path.basename(jpg_path)] = name
            else:
                with Image.open(target_path) as img_obj:
                    if "exif" in img_obj.info:
                        metadata_cache[os.path.basename(target_path)] = img_obj.info["exif"]
                processed_files.append(target_path)
                original_filenames[os.path.basename(target_path)] = name

            # Display image info
            with Image.open(target_path) as img:
                mp = (img.width * img.height) / 1000000
                print(f"   ✅ Loaded: {os.path.basename(target_path)} | {img.width}x{img.height} ({mp:.2f} MP)")

        except Exception as e:
            print(f"   ❌ Failed to process {name}: {str(e)}")
            continue

    # Clear uploads from memory
    del uploaded
    gc.collect()

    if not processed_files:
        print("❌ No valid images were uploaded or processed.")
    else:
        # Prepare Real-ESRGAN command
        face_flag = "--face_enhance" if enhance_faces else ""
        ai_scale = min(4, upscale_factor) if upscale_factor > 4 else upscale_factor

        print("\n🤖 --- STAGE 2: REAL-ESRGAN AI ENGINE ---")
        print(f"⚙️ AI upscale factor: {ai_scale}x")

        # Build command as list for safe execution
        run_cmd = [
            "python", "inference_realesrgan.py",
            "-n", "RealESRGAN_x4plus",
            "-i", "inputs",
            "-o", "results",
            "--outscale", str(ai_scale),
            "--tile", str(tile_size if tile_size > 0 else 512),
            "--tile_pad", "10"
        ]
        if face_flag:
            run_cmd.append(face_flag)

        print(f"🖥️ Running Real-ESRGAN...")

        # Execute with better error handling
        try:
            process = subprocess.run(run_cmd, capture_output=True, text=True, timeout=3600)

            if process.returncode != 0:
                print(f"\n⚠️ Primary execution had issues. Trying fallback mode...")
                # Print stderr for debugging
                if process.stderr:
                    print(f"Error: {process.stderr[:500]}")
                # Fallback with smaller tile size
                fallback_cmd = run_cmd.copy()
                for i, arg in enumerate(fallback_cmd):
                    if arg == "--tile":
                        fallback_cmd[i+1] = "256"
                        break

                fallback_process = subprocess.run(fallback_cmd, capture_output=True, text=True, timeout=3600)
                if fallback_process.returncode != 0:
                    print(f"❌ Fallback also failed!")
                    if fallback_process.stderr:
                        print(f"Error: {fallback_process.stderr[:500]}")
                    raise Exception("Real-ESRGAN processing failed")
                else:
                    print("✅ Fallback mode succeeded!")

        except subprocess.TimeoutExpired:
            print("❌ Processing timed out after 1 hour")
            raise
        except Exception as e:
            print(f"❌ Failed to run Real-ESRGAN: {str(e)}")
            raise

        print(f"\n📐 --- STAGE 3: POST-PROCESSING & METADATA INJECTION ---")

        results_files = glob.glob('results/*')

        if not results_files:
            print("❌ No output files generated by Real-ESRGAN")
        else:
            # Define resampling methods
            pil_methods = {
                "Lanczos4": Image.Resampling.LANCZOS,
                "Bicubic": Image.Resampling.BICUBIC,
                "Linear": Image.Resampling.BILINEAR
            }

            # Resolution limits mapping
            res_mapping = {"4K": 3840, "8K": 7680, "12K": 11520, "16K": 15360}
            max_edge = res_mapping.get(max_edge_resolution, None) if max_edge_resolution != "No Limit" else None

            final_images = []

            for idx, path in enumerate(results_files, 1):
                try:
                    print(f" ⏳ [{idx}/{len(results_files)}] Processing: {os.path.basename(path)}")

                    # Open image
                    img = Image.open(path)
                    img_final = img
                    out_filename = os.path.basename(path)

                    # Apply additional upscaling if needed
                    if upscale_factor > 4:
                        pil_scale = upscale_factor / 4.0
                        new_width = int(img.width * pil_scale)
                        new_height = int(img.height * pil_scale)
                        img_final = img.resize((new_width, new_height), resample=pil_methods[opencv_method])
                        # Change output filename to reflect total upscale factor
                        base_name = os.path.splitext(out_filename)[0]
                        base_name = base_name.replace('_out', '')
                        out_filename = f"{base_name}_{upscale_factor}x.jpg"
                    elif upscale_factor < 4:
                        # Downscaling if needed
                        down_scale = upscale_factor / 4.0
                        new_width = int(img.width * down_scale)
                        new_height = int(img.height * down_scale)
                        img_final = img.resize((new_width, new_height), resample=pil_methods[opencv_method])
                        base_name = os.path.splitext(out_filename)[0]
                        base_name = base_name.replace('_out', '')
                        out_filename = f"{base_name}_{upscale_factor}x.jpg"
                    else:
                        # Just remove '_out' suffix for 4x upscale
                        out_filename = out_filename.replace('_out', '')

                    # Apply resolution limit
                    if max_edge:
                        w, h = img_final.size
                        if max(w, h) > max_edge:
                            if w > h:
                                new_w = max_edge
                                new_h = int(h * (max_edge / w))
                            else:
                                new_h = max_edge
                                new_w = int(w * (max_edge / h))
                            img_final = img_final.resize((new_w, new_h), Image.Resampling.LANCZOS)
                            print(f"   📏 Resized to {new_w}x{new_h} (max {max_edge_resolution})")

                    # Apply sharpness
                    if apply_sharpness and sharpness_intensity > 0:
                        try:
                            # Convert PIL to OpenCV
                            cv_img = cv2.cvtColor(np.array(img_final), cv2.COLOR_RGB2BGR)

                            # Apply unsharp mask
                            gaussian_blur = cv2.GaussianBlur(cv_img, (0, 0), sigmaX=1.0)
                            cv_sharpened = cv2.addWeighted(cv_img, 1.0 + sharpness_intensity,
                                                          gaussian_blur, -sharpness_intensity, 0)

                            # Clip values to valid range
                            cv_sharpened = np.clip(cv_sharpened, 0, 255).astype(np.uint8)
                            img_final = Image.fromarray(cv2.cvtColor(cv_sharpened, cv2.COLOR_BGR2RGB))
                        except Exception as e:
                            print(f"   ⚠️ Sharpening failed: {e}, continuing without sharpening")

                    # Restore metadata - try multiple lookup methods
                    exif_data = b""

                    # Method 1: Direct lookup with original filename (without _out)
                    orig_filename = os.path.basename(path).replace('_out', '')
                    if orig_filename in metadata_cache:
                        exif_data = metadata_cache[orig_filename]
                    else:
                        # Method 2: Try finding by comparing base names
                        for cached_name in metadata_cache.keys():
                            # Remove extension for comparison
                            cached_base = os.path.splitext(cached_name)[0]
                            orig_base = os.path.splitext(orig_filename)[0]
                            if cached_base == orig_base:
                                exif_data = metadata_cache[cached_name]
                                break
                        else:
                            # Method 3: Use the first available metadata if any exists
                            if metadata_cache:
                                exif_data = list(metadata_cache.values())[0]
                                print(f"   ℹ️ Using fallback metadata from another file")

                    # Save final image
                    final_dest = os.path.join(output_folder, out_filename)

                    # Save with metadata if available
                    if exif_data and len(exif_data) > 0:
                        img_final.save(final_dest, quality=95, optimize=True, exif=exif_data)
                    else:
                        img_final.save(final_dest, quality=95, optimize=True)
                        print(f"   ℹ️ No metadata to preserve")

                    final_images.append(final_dest)
                    print(f"   ✅ Saved: {out_filename}")

                    # Clear from memory
                    del img, img_final

                except Exception as e:
                    print(f"   ❌ Failed to process {path}: {str(e)}")
                    import traceback
                    traceback.print_exc()
                    continue

            print(f"\n📦 --- STAGE 4: PACKAGING & DOWNLOAD ---")

            if final_images:
                print(f"✅ Successfully processed {len(final_images)} image(s)")

                # Create ZIP archive
                zip_name = f'/content/Enhanced_{upscale_factor}x_{len(final_images)}images'
                try:
                    shutil.make_archive(zip_name, 'zip', output_folder)

                    # Download ZIP file
                    zip_path = f'{zip_name}.zip'
                    if os.path.exists(zip_path):
                        files.download(zip_path)
                        print(f"🎉 Pipeline completed successfully!")
                        print(f"📁 Output saved to: {output_folder}")
                    else:
                        print("❌ Failed to create ZIP archive")

                except Exception as e:
                    print(f"❌ Error creating ZIP: {str(e)}")
                    print("Files are still available in the output folder")
            else:
                print("❌ No images were successfully processed")

            # Cleanup
            gc.collect()
            if torch.cuda.is_available():
                torch.cuda.empty_cache()

📦 Installing opencv-python...
/content/Real-ESRGAN
🔄 Clearing PyTorch CUDA cache...

📁 Select your image(s):


Saving mouse.jpg to mouse (1).jpg

📊 --- STAGE 1: DISK-STREAM INGESTION & METADATA CACHE ---
   ✅ Loaded: mouse_1.jpg | 800x450 (0.36 MP)

🤖 --- STAGE 2: REAL-ESRGAN AI ENGINE ---
⚙️ AI upscale factor: 4x
🖥️ Running Real-ESRGAN...

📐 --- STAGE 3: POST-PROCESSING & METADATA INJECTION ---
 ⏳ [1/1] Processing: mouse_1_out.jpg
   ℹ️ No metadata to preserve
   ✅ Saved: mouse_1_6x.jpg

📦 --- STAGE 4: PACKAGING & DOWNLOAD ---
✅ Successfully processed 1 image(s)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

🎉 Pipeline completed successfully!
📁 Output saved to: /content/output


In [ ]:
import zipfile

# Zip the results folder
shutil.make_archive('enhanced_results', 'zip', 'results')

# Download the zip file
files.download('enhanced_results.zip')
print("Download started! Check your browser downloads.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download started! Check your browser downloads.
